# Appendix: NUTS + NN-Surrogate Inference (settled, intractable on M1 CPU)

This notebook was the planned full-scale validation of the NN-surrogate NUTS + EKF pipeline ([08a](08a_pretrain_nn_surrogate.ipynb) bundle, [`make_neural_bayesian_spec`](../src/v2/estimation/bayesian_basic_investment.py)). The closed-form NUTS counterpart ran cleanly in [08c](08c_nuts_closedform_validation.ipynb); this run swaps the analytical policy for the cached SHAC NN inside the same EKF.

**Status.** The CPU_LARGE attempt (4 chains × 50 firms × 20 periods, 1000 warmup + 500 samples) ran for 40+ hours on an Apple M1 before being interrupted unfinished. Per-leapfrog cost is dominated by the NN forward + reverse-mode Jacobian at each EKF step; with NUTS's ~30-100 leapfrog steps per iteration the budget compounds beyond any practical CPU wall.

**Verdict.** NUTS-HMC on the cached NN policy is **not tractable on M1 CPU at production sample sizes**. The closed-form 08c run is the validated NUTS reference; gradient-free RW-MH/RAM on the same NN ([08b](08b_rwmh_three_way_baseline.ipynb)) is the recommended NN-surrogate path on this hardware. The notebook is kept here as a frozen record of the attempt and the cost calibration.

**What is reproducible.** Only the `CPU_CAL` profile completes in a practical wall (~8 min). It is a per-iter cost calibration, not a usable posterior. 2 chains and 100 samples are insufficient for inference. Larger profiles are retained for documentation only and should not be run without a GPU.

## Section 0: Setup


In [ ]:
from pathlib import Path
import json
import os
import sys
import time
import warnings

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
warnings.filterwarnings("ignore", message=r".*does not produce the same series as CPU implementation.*")
warnings.filterwarnings("ignore", message=r".*casting an input of type complex64.*")

_REPO_ROOT = None
for _rr_candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_rr_candidate / "src" / "v2").exists():
        _REPO_ROOT = _rr_candidate
        break
if _REPO_ROOT is None:
    import importlib.util as _rr_ilu
    _rr_spec = _rr_ilu.find_spec("src.v2")
    if _rr_spec is not None and _rr_spec.origin is not None:
        _REPO_ROOT = Path(_rr_spec.origin).resolve().parents[2]
if _REPO_ROOT is None:
    raise RuntimeError("Could not find repo root containing src/v2.")
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt

import tensorflow as tf
tf.config.set_visible_devices([], "GPU")
tf.get_logger().setLevel("ERROR")

from src.v2.environments.basic_investment import EconomicParams, ShockParams
from src.v2.environments.parameterized_basic_investment import ParameterizedBasicInvestmentEnv
from src.v2.estimation import BayesianRunConfig, run_mcmc
from src.v2.estimation.bayesian_basic_investment import (
    load_policy_with_normalizer,
    make_neural_bayesian_spec,
)
from src.v2.estimation.beta_sampler import DEFAULT_UNIFORM_BOUNDS
from src.v2.networks.policy import ParameterizedPolicyNetwork
from src.v2.utils.seeding import fold_in_seed

print(f"tf: {tf.__version__}, devices: {[d.device_type for d in tf.config.list_logical_devices()]}")
np.set_printoptions(precision=4, suppress=True)

# ---- Load bundle from 08a -------------------------------------------------
BUNDLE_BASE = _REPO_ROOT / "outputs" / "notebooks" / "08a_pretrain_nn_surrogate" / "shac_full_frictionless"
if not BUNDLE_BASE.with_suffix(".meta.json").exists():
    raise FileNotFoundError(f"Bundle not found at {BUNDLE_BASE}.*. Run 08a first.")

meta = json.loads(BUNDLE_BASE.with_suffix(".meta.json").read_text())
panel_full = dict(np.load(BUNDLE_BASE.with_suffix(".panel.npz")))

param_env = ParameterizedBasicInvestmentEnv(
    nominal_econ=EconomicParams(**meta["env"]["nominal_econ"]),
    nominal_shocks=ShockParams(**meta["env"]["nominal_shocks"]),
    bounds=meta["env"]["bounds"],
)
policy_nn = ParameterizedPolicyNetwork(
    state_dim=param_env.state_dim(), action_dim=param_env.action_dim(), beta_dim=5,
    **param_env.action_spec(),
    n_layers=meta["nn_architecture"]["n_layers"],
    n_neurons=meta["nn_architecture"]["n_neurons"], seed=(202, 0),
)
policy_nn(tf.zeros((1, param_env.state_dim())), tf.zeros((1, 5)))
load_policy_with_normalizer(policy_nn, BUNDLE_BASE.with_suffix(".weights.h5"))
policy_nn.trainable = False

beta_true   = meta["ground_truth_beta"]
MASTER_SEED = tuple(meta["master_seed"])

OUTPUT_DIR = _REPO_ROOT / "outputs" / "notebooks" / "appendix_08_nuts_nn_validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Bundle loaded: panel y={panel_full['y'].shape}, ground-truth beta = {beta_true}")
print(f"OUTPUT_DIR = {OUTPUT_DIR}")


## Section 1: Profile, spec, and observations

Default profile is `CPU_CAL` (calibration only, not a usable posterior). `CPU_SMOKE` is retained for documentation; it does not complete on M1 CPU within a reasonable wall.


In [ ]:
# CPU_CAL is the only profile that completes on M1 CPU within a usable wall.
# CPU_SMOKE/CPU_LARGE entries are kept as documentation of the planned A/B vs 08c;
# in practice the NN per-leapfrog cost makes anything beyond CAL intractable on M1.
MODE = "CPU_CAL"
PROFILES = {
    "CPU_CAL":   dict(n_chains=2, n_warmup=300,  n_samples=100,
                       n_firms=10, horizon=5,  target_accept_prob=0.90),  # ~8 min on M1
    "CPU_SMOKE": dict(n_chains=4, n_warmup=1000, n_samples=500,
                       n_firms=20, horizon=10, target_accept_prob=0.85),  # est. >24 h, untested to completion
}
P       = PROFILES[MODE]
RUN_DIR = OUTPUT_DIR / MODE
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"MODE={MODE}; batch={P['n_chains'] * P['n_firms'] * P['horizon']}; RUN_DIR={RUN_DIR}")


In [ ]:
spec = make_neural_bayesian_spec(param_env, policy_nn)

# Slice the bundle's panel to the per-profile shape. The bundle already
# contains (y, log_k, log_k_next) with eta and xi baked in at BETA_TRUE
# (08a Section 2), so no extra noise is added here.
N_slice, T_slice = int(P["n_firms"]), int(P["horizon"])
observed = {
    "y":          panel_full["y"][:N_slice,          :T_slice, np.newaxis].astype(np.float32),
    "log_k":      panel_full["log_k"][:N_slice,      :T_slice, np.newaxis].astype(np.float32),
    "log_k_next": panel_full["log_k_next"][:N_slice, :T_slice, np.newaxis].astype(np.float32),
}
print(f"Panel slice: {N_slice} firms x {T_slice} model-time periods")

priors = {
    "alpha":           f"Uniform{DEFAULT_UNIFORM_BOUNDS['alpha']}",
    "rho":             f"Uniform{DEFAULT_UNIFORM_BOUNDS['rho']}",
    "sigma_epsilon":   f"Uniform{DEFAULT_UNIFORM_BOUNDS['sigma_epsilon']}",
    "production_mean": "Normal(0, 0.1)",
    "production_std":  "HalfNormal(0.5)",
    "investment_mean": "Normal(0, 0.1)",
    "investment_std":  "HalfNormal(0.5)",
}
prior_table = pd.DataFrame([
    {"parameter": k, "true": beta_true[k], "prior": priors[k]} for k in spec.parameter_names
])
display(prior_table)
prior_table.to_csv(RUN_DIR / "prior_table.csv", index=False)


## Section 2: MCMC

NUTS via `tfp.experimental.mcmc.windowed_adaptive_nuts`. Per-leapfrog cost is dominated by the NN forward + reverse-mode Jacobian at each EKF step; this is the bottleneck that makes profiles beyond `CPU_CAL` intractable on M1 CPU.


In [ ]:
mcmc_seed = fold_in_seed(MASTER_SEED, "mcmc")
cfg = BayesianRunConfig(
    n_chains=P["n_chains"], n_warmup=P["n_warmup"], n_samples=P["n_samples"],
    target_accept_prob=P["target_accept_prob"], master_seed=tuple(MASTER_SEED),
)
print(f"Running MCMC: {cfg.n_chains} chains x ({cfg.n_warmup} warmup + {cfg.n_samples} samples)...")
t0 = time.time()
result = run_mcmc(spec, observed, cfg, seed=tuple(mcmc_seed))
mcmc_wall_sec = time.time() - t0
md_meta = result.metadata
print(f"MCMC wall = {mcmc_wall_sec:.1f}s   acceptance = {result.acceptance_rate:.3f}")
print(f"Mean leapfrog steps per iter: {md_meta['mean_leapfrog_steps']:.1f}  "
      f"(max {md_meta['max_leapfrog_steps']}, divergences {md_meta['divergence_count']})")

Running MCMC: 4 chains x (1000 warmup + 500 samples)...


I0000 00:00:1779698053.406024 3634496 service.cc:145] XLA service 0x38be67c50 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779698053.406049 3634496 service.cc:153]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1779698057.107499 3634501 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


## Section 3: Report

Posterior summary, trace, marginals, and posterior correlation. At `CPU_CAL` the posterior is **not** trustworthy (2 chains, 100 samples); read this section as a runtime calibration, not an inference result.


In [ ]:
rows = []
for name in spec.parameter_names:
    s = result.posterior_samples[name]
    lo, hi = float(np.quantile(s, 0.025)), float(np.quantile(s, 0.975))
    rows.append({
        "parameter": name,
        "true":      beta_true[name],
        "median":    float(np.median(s)),
        "ci_2.5":    lo,
        "ci_97.5":   hi,
        "r_hat":     float(result.r_hat[name]),
        "ess":       float(result.ess[name]),
        "pass":      bool(lo <= beta_true[name] <= hi),
    })
posterior_table = pd.DataFrame(rows)
print("Posterior summary:")
display(posterior_table)
posterior_table.to_csv(RUN_DIR / "posterior_summary.csv", index=False)

# Persist raw samples and run metadata so re-analysis never requires re-running MCMC.
np.savez(RUN_DIR / "posterior_samples.npz",
         **{name: np.asarray(result.posterior_samples[name]) for name in spec.parameter_names})
run_metadata = {
    "mode":                MODE,
    "profile":             P,
    "wall_sec":            float(mcmc_wall_sec),
    "acceptance_rate":     float(result.acceptance_rate),
    "mean_leapfrog_steps": float(md_meta["mean_leapfrog_steps"]),
    "max_leapfrog_steps":  int(md_meta["max_leapfrog_steps"]),
    "divergence_count":    int(md_meta["divergence_count"]),
    "master_seed":         list(MASTER_SEED),
    "n_pass":              int(posterior_table["pass"].sum()),
    "n_params":            len(spec.parameter_names),
}
(RUN_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2))
print(f"\nSaved → {RUN_DIR}:")
print(f"  posterior_summary.csv, posterior_samples.npz, run_metadata.json")
print(f"  (trace.png, marginals.png, posterior_correlation.png saved by following cells)")
print(f"\nPass: {posterior_table['pass'].sum()}/{len(spec.parameter_names)} params have truth in 95% CI")

In [ ]:
fig, axes = plt.subplots(len(spec.parameter_names), 1,
                          figsize=(9, 1.8 * len(spec.parameter_names)), sharex=True)
for ax, name in zip(axes, spec.parameter_names):
    samples = result.posterior_samples[name]
    for c in range(samples.shape[1]):
        ax.plot(samples[:, c], lw=0.4, alpha=0.8, label=f"chain {c}")
    ax.axhline(beta_true[name], color="k", lw=1.0, ls="--", label="true")
    ax.set_ylabel(name)
    ax.legend(loc="upper right", fontsize=8, frameon=False)
axes[-1].set_xlabel("post-warmup sample index")
plt.suptitle(f"Trace (MODE={MODE})")
plt.tight_layout()
plt.savefig(RUN_DIR / "trace.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(spec.parameter_names),
                          figsize=(3.0 * len(spec.parameter_names), 3))
for ax, name in zip(axes, spec.parameter_names):
    samples = result.posterior_samples[name].reshape(-1)
    ax.hist(samples, bins=40, alpha=0.7, density=True, color="steelblue")
    ax.axvline(beta_true[name], color="k", lw=1.5, ls="--")
    ax.axvline(np.median(samples), color="red", lw=1.0)
    ax.set_title(name); ax.set_yticks([])
plt.suptitle(f"Posterior marginals (dashed=true, red=median; MODE={MODE})")
plt.tight_layout()
plt.savefig(RUN_DIR / "marginals.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# Posterior correlation across the 7 sampled params. Strong negative
# Corr(alpha, investment_mean) reflects the structural ridge kappa(alpha) + mu_xi = const
# in the likelihood (Bayesian.md Section 1.2). At CPU_CAL this is unstable;
# treat as a runtime artifact, not a posterior summary.

samples_flat = {name: result.posterior_samples[name].reshape(-1) for name in spec.parameter_names}
samples_df = pd.DataFrame(samples_flat)
corr = samples_df.corr()

print("Posterior correlation matrix (NN-surrogate, CPU_CAL):")
display(corr.round(2))

r_alpha_inv = corr.loc["alpha", "investment_mean"]
print(f"\nCorr(alpha, investment_mean) = {r_alpha_inv:+.3f}")

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.index)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iat[i, j]:.2f}", ha="center", va="center",
                color="white" if abs(corr.iat[i, j]) > 0.5 else "black", fontsize=8)
plt.colorbar(im, ax=ax, label="Posterior correlation")
plt.title(f"Posterior correlation (NN-surrogate, MODE={MODE})")
plt.tight_layout()
plt.savefig(RUN_DIR / "posterior_correlation.png", dpi=140, bbox_inches="tight")
plt.show()
